<a href="https://colab.research.google.com/github/Talzablev/Turtels/blob/main/tirgul4_cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px # ספרייה מומלצת לגרפים אינטראקטיביים

# 1. טעינת הנתונים (החלף את ה-URL בקישור האמיתי של משרד התחבורה במידת הצורך)
# דוגמה לטעינה בסיסית:
# url = "https://data.gov.il/api/3/action/datastore_search?resource_id=053ad243-5e8b-4480-99c5-dc063856943d"
# df = pd.read_csv(url)

# לצורך הדוגמה, ניצור נתונים פיקטיביים:
data = {
    'shnat_yitzur': [2020, 2021, 2020, 2022, 2021, 2023],
    'tozeret_nm': ['Toyota', 'Mazda', 'Toyota', 'Tesla', 'Mazda', 'Tesla'],
    'degem_nm': ['Corolla', '3', 'Yaris', 'Model 3', 'CX-5', 'Model Y']
}
df = pd.DataFrame(data)

# יצירת Output widgets לכל טאב כדי לשלוט בתצוגה
out1 = widgets.Output()
out2 = widgets.Output()
out3 = widgets.Output()

# --- טאב 1: סטטיסטיקה ---
with out1:
    print("סטטיסטיקה של המאגר:")
    display(df.describe())

# --- טאב 2: הצגת המידע ---
with out2:
    print("כלל הנתונים בטבלה:")
    display(df)

# --- טאב 3: גרף ---
with out3:
    # יצירת גרף המציג כמות מכוניות לפי שנת ייצור
    fig = px.histogram(df, x="shnat_yitzur", title="מספר מכוניות לפי שנת ייצור")
    fig.show()

# --- בניית הטאבים ---
tabs = widgets.Tab(children=[out1, out2, out3])
tabs.set_title(0, 'סטטיסטיקה')
tabs.set_title(1, 'הצגת מידע')
tabs.set_title(2, 'גרף שנתי')

display(tabs)

In [2]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px

# 1. יצירת דאטה-סט לדוגמה (בפרויקט האמיתי תטען את ה-CSV מהמאגר הממשלתי)
data = {
    'tozeret_nm': ['TOYOTA', 'TOYOTA', 'MAZDA', 'MAZDA', 'TESLA', 'TESLA', 'HYUNDAI'],
    'degem_nm': ['COROLLA', 'YARIS', '3', 'CX-5', 'MODEL 3', 'MODEL Y', 'IONIQ'],
    'shnat_yitzur': [2020, 2021, 2019, 2022, 2023, 2022, 2021],
    'ramat_gimur': ['SUN', 'STYLE', 'COMFORT', 'EXECUTIVE', 'RWD', 'LONG RANGE', 'PREMIUM']
}
df = pd.DataFrame(data)

# 2. הגדרת רכיבי ממשק (Widgets)
out_stats = widgets.Output()
out_data = widgets.Output()
out_graph = widgets.Output()
out_filter = widgets.Output()

# רכיבי סינון לטאב הבונוס
dropdown_maker = widgets.Dropdown(options=sorted(df['tozeret_nm'].unique()), description='יצרן:')
dropdown_model = widgets.Dropdown(description='דגם:')
btn_filter = widgets.Button(description="הצג נתונים", button_style='success')

# 3. פונקציות לוגיקה
def update_models(*args):
    """עדכון רשימת הדגמים לפי היצרן שנבחר"""
    filtered_models = df[df['tozeret_nm'] == dropdown_maker.value]['degem_nm'].unique()
    dropdown_model.options = sorted(filtered_models)

dropdown_maker.observe(update_models, 'value')
update_models() # הרצה ראשונית

def on_button_clicked(b):
    with out_filter:
        clear_output()
        res = df[(df['tozeret_nm'] == dropdown_maker.value) & (df['degem_nm'] == dropdown_model.value)]
        print(f"נמצאו {len(res)} רכבים מדגם זה.")
        print("רמות גימור קיימות:")
        print(res['ramat_gimur'].to_list())
        display(res)

btn_filter.on_click(on_button_clicked)

# 4. מילוי הטאבים בתוכן
with out_stats:
    display(df.describe())

with out_data:
    display(df)

with out_graph:
    fig = px.histogram(df, x="shnat_yitzur", title="התפלגות רכבים לפי שנת ייצור",
                       labels={'shnat_yitzur': 'שנת ייצור'}, color_discrete_sequence=['#636EFA'])
    fig.show()

with out_filter:
    display(widgets.VBox([dropdown_maker, dropdown_model, btn_filter]))

# 5. יצירת הטאבים והצגתם
tabs = widgets.Tab(children=[out_stats, out_data, out_graph, out_filter])
tabs.set_title(0, 'סטטיסטיקה')
tabs.set_title(1, 'כל המידע')
tabs.set_title(2, 'גרף שנתי')
tabs.set_title(3, 'חיפוש מתקדם (בונוס)')

display(tabs)

In [7]:
import pandas as pd
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px

# --- שלב 1: שליפת נתונים מהענן (Gov Data API) --- כלול טיפול בשגיאות משופר
def get_car_data(limit=1000):
    resource_id = "053ad243-5e8b-4480-99c5-dc063856943d" # מזהה מאגר רכב פעיל
    url = f"https://data.gov.il/api/3/action/datastore_search?resource_id={resource_id}&limit={limit}"

    try:
        response = requests.get(url)
        response.raise_for_status() # העלה חריגה עבור קודי סטטוס שגויים (4xx או 5xx)
        res_json = response.json()

        # ודא שהמבנה הצפוי קיים בתגובה
        if 'result' in res_json and 'records' in res_json['result']:
            records = res_json['result']['records']
            # המרה ראשונית ל-DataFrame כאן, כדי שנוכל לבדוק עמודות
            df_temp = pd.DataFrame(records)

            # בדיקה אם קיימות עמודות חיוניות לפני החזרה
            required_cols = ['shnat_yitzur', 'tozeret_nm', 'degem_nm']
            if all(col in df_temp.columns for col in required_cols):
                return df_temp
            else:
                print(f"שגיאה: חסרות עמודות חיוניות בנתונים שהתקבלו. נדרשות: {', '.join(required_cols)}")
                return pd.DataFrame()
        else:
            print("שגיאה: מבנה התגובה מה-API אינו כצפוי. חסר מפתח 'result' או 'records'.")
            return pd.DataFrame()
    except requests.exceptions.HTTPError as errh:
        print(f"שגיאת HTTP בעת משיכת נתונים: {errh}")
        return pd.DataFrame()
    except requests.exceptions.ConnectionError as errc:
        print(f"שגיאת התחברות בעת משיכת נתונים: {errc}")
        return pd.DataFrame()
    except requests.exceptions.Timeout as errt:
        print(f"שגיאת Timeout בעת משיכת נתונים: {errt}")
        return pd.DataFrame()
    except requests.exceptions.RequestException as err:
        print(f"שגיאה כללית במשיכת נתונים: {err}")
        return pd.DataFrame()
    except Exception as e: # לתפוס שגיאות לא צפויות אחרות, למשל שגיאות ניתוח JSON
        print(f"שגיאה לא צפויה במשיכת נתונים: {e}")
        return pd.DataFrame()

# טעינת הנתונים
print("מושך נתונים מהענן... נא להמתין")
df_from_api = get_car_data(2000) # נמשוך 2000 רשומות לדוגמה

# אם קיימת שגיאה במשיכת נתונים, נשתמש בדאטה-סט לדוגמה
if df_from_api.empty:
    print("שגיאה במשיכת נתונים מה-API, משתמשים בדאטה-סט לדוגמה.")
    data = {
        'tozeret_nm': ['TOYOTA', 'TOYOTA', 'MAZDA', 'MAZDA', 'TESLA', 'TESLA', 'HYUNDAI', 'HONDA', 'BMW', 'MERCEDES'],
        'degem_nm': ['COROLLA', 'YARIS', '3', 'CX-5', 'MODEL 3', 'MODEL Y', 'IONIQ', 'CIVIC', 'X5', 'C-CLASS'],
        'shnat_yitzur': [2020, 2021, 2019, 2022, 2023, 2022, 2021, 2020, 2023, 2021],
        'ramat_gimur': ['SUN', 'STYLE', 'COMFORT', 'EXECUTIVE', 'RWD', 'LONG RANGE', 'PREMIUM', 'LX', 'SPORT', 'AVANTGARDE']
    }
    df = pd.DataFrame(data)
else:
    df = df_from_api.copy()

# ניקוי בסיסי - המרת שנת ייצור למספר (רק אם ה-DataFrame אינו ריק)
if not df.empty and 'shnat_yitzur' in df.columns:
    df['shnat_yitzur'] = pd.to_numeric(df['shnat_yitzur'], errors='coerce')
else:
    print("לא נטענו נתונים או שהעמודה 'shnat_yitzur' אינה קיימת. ה-DataFrame ריק או חסר.")

# --- שלב 2: הגדרת הממשק (Widgets) ---
out_stats = widgets.Output()
out_table = widgets.Output()
out_graph = widgets.Output()
out_search = widgets.Output()

# מילוי הטאבים בתוכן רק אם יש נתונים
if not df.empty:
    # --- טאב 1: סטטיסטיקה ---
    with out_stats:
        display(df.describe())

    # --- טאב 2: טבלה ---
    with out_table:
        display(df.head(100)) # מציג רק 100 ראשונות לביצועים מהירים

    # --- טאב 3: גרף ---
    with out_graph:
        # גרף התפלגות שנת ייצור
        fig = px.histogram(df, x="shnat_yitzur", title="מספר רכבים לפי שנת ייצור (מתוך המדגם)")
        fig.show()

    # --- טאב 4: חיפוש דינמי (הפיצ'ר הנוסף) ---
    # וודא שהעמודות קיימות לפני יצירת הרכיבים
    if 'tozeret_nm' in df.columns and 'degem_nm' in df.columns:
        maker_dropdown = widgets.Dropdown(options=sorted(df['tozeret_nm'].unique()), description='יצרן:')
        model_dropdown = widgets.Dropdown(description='דגם:')

        def update_models(*args):
            relevant_models = df[df['tozeret_nm'] == maker_dropdown.value]['degem_nm'].unique()
            model_dropdown.options = sorted(relevant_models)

        maker_dropdown.observe(update_models, 'value')
        update_models()

        search_btn = widgets.Button(description="חפש במאגר", button_style='info')

        def perform_search(b):
            with out_search:
                clear_output()
                display(widgets.VBox([maker_dropdown, model_dropdown, search_btn]))
                filtered = df[(df['tozeret_nm'] == maker_dropdown.value) & (df['degem_nm'] == model_dropdown.value)]
                print(f"נמצאו {len(filtered)} רכבים.")
                display(filtered)

        search_btn.on_click(perform_search)

        with out_search:
            display(widgets.VBox([maker_dropdown, model_dropdown, search_btn]))
    else:
        with out_search:
            print("אין נתונים מספיקים עבור חיפוש מתקדם (חסרות עמודות 'tozeret_nm' או 'degem_nm').")

else: # אם df ריק, הצג הודעה בכל ווידג'ט פלט
    with out_stats:
        print("אין נתונים זמינים לסטטיסטיקה.")
    with out_table:
        print("אין נתונים זמינים להצגה.")
    with out_graph:
        print("אין נתונים זמינים לגרף.")
    with out_search:
        print("אין נתונים זמינים לחיפוש מתקדם.")

# --- שלב 3: תצוגת הטאבים ---
tabs = widgets.Tab(children=[out_stats, out_table, out_graph, out_search])
tabs.set_title(0, 'סטטיסטיקה')
tabs.set_title(1, 'נתונים גולמיים')
tabs.set_title(2, 'ויזואליזציה')
tabs.set_title(3, 'חיפוש מתקדם')

display(tabs)

מושך נתונים מהענן... נא להמתין
שגיאת HTTP בעת משיכת נתונים: 404 Client Error: Not Found for url: https://data.gov.il/api/3/action/datastore_search?resource_id=053ad243-5e8b-4480-99c5-dc063856943d&limit=2000
שגיאה במשיכת נתונים מה-API, משתמשים בדאטה-סט לדוגמה.


In [8]:
import pandas as pd
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. משיכת נתונים מה-API (לפי דוגמת ה-JSON שלמדנו) ---
def fetch_gov_data(limit=5000):
    resource_id = "053ad243-5e8b-4480-99c5-dc063856943d"
    url = f"https://data.gov.il/api/3/action/datastore_search?resource_id={resource_id}&limit={limit}"

    try:
        response = requests.get(url)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        data = response.json()

        if 'result' in data and 'records' in data['result']:
            return pd.DataFrame(data['result']['records'])
        else:
            print("שגיאה: מבנה התגובה מה-API אינו כצפוי. חסר מפתח 'result' או 'records'.")
            return pd.DataFrame() # Return empty DataFrame on unexpected structure
    except requests.exceptions.RequestException as e:
        print(f"שגיאה בעת משיכת נתונים מה-API: {e}")
        return pd.DataFrame() # Return empty DataFrame on request error
    except ValueError as e: # Catches JSONDecodeError or other JSON parsing issues
        print(f"שגיאה בניתוח תגובת ה-API כ-JSON: {e}")
        return pd.DataFrame()

print("מתחבר לענן ושולף נתונים...")
df_from_api = fetch_gov_data()

# אם קיימת שגיאה במשיכת נתונים, נשתמש בדאטה-סט לדוגמה
if df_from_api.empty:
    print("שגיאה במשיכת נתונים מה-API, משתמשים בדאטה-סט לדוגמה.")
    data = {
        'tozeret_nm': ['TOYOTA', 'TOYOTA', 'MAZDA', 'MAZDA', 'TESLA', 'TESLA', 'HYUNDAI', 'HONDA', 'BMW', 'MERCEDES'],
        'degem_nm': ['COROLLA', 'YARIS', '3', 'CX-5', 'MODEL 3', 'MODEL Y', 'IONIQ', 'CIVIC', 'X5', 'C-CLASS'],
        'shnat_yitzur': [2020, 2021, 2019, 2022, 2023, 2022, 2021, 2020, 2023, 2021],
        'ramat_gimur': ['SUN', 'STYLE', 'COMFORT', 'EXECUTIVE', 'RWD', 'LONG RANGE', 'PREMIUM', 'LX', 'SPORT', 'AVANTGARDE']
    }
    df = pd.DataFrame(data)
else:
    df = df_from_api.copy()

# הכנת נתונים בסיסית (המרה למספרים עבור הגרפים)
if not df.empty and 'shnat_yitzur' in df.columns:
    df['shnat_yitzur'] = pd.to_numeric(df['shnat_yitzur'], errors='coerce')
else:
    print("לא נטענו נתונים או שהעמודה 'shnat_yitzur' אינה קיימת. ה-DataFrame ריק או חסר.")


# --- 2. יצירת הטאבים (לפי המבנה במצגת) ---
tab = widgets.Tab()
out1 = widgets.Output()
out2 = widgets.Output()
out3 = widgets.Output()

tab.children = [out1, out2, out3]
tab.set_title(0, 'Data Overview')
tab.set_title(1, 'Raw Data')
tab.set_title(2, 'Year Count')

# --- טאב 1: סטטיסטיקה (מצגת: Tab 1) ---
with out1:
    print("Data Overview:")
    if not df.empty:
        # שימוש ב-describe כפי שמופיע בצילום המסך במצגת
        display(df.describe())

        # הוספת גרף התפלגות (Distribution) כפי שמופיע במצגת
        if 'shnat_yitzur' in df.columns:
            plt.figure(figsize=(10, 5))
            sns.histplot(df['shnat_yitzur'].dropna(), kde=True, color='blue')
            plt.title('Distribution of Year')
            plt.xlabel('shnat_yitzur')
            plt.ylabel('Frequency')
            plt.show()
        else:
            print("העמודה 'shnat_yitzur' אינה קיימת עבור גרף התפלגות.")
    else:
        print("אין נתונים זמינים לסטטיסטיקה או לגרף התפלגות.")

# --- טאב 2: נתונים גולמיים (מצגת: Tab 2) ---
with out2:
    print("Raw Data (First 100 rows):")
    if not df.empty:
        display(df.head(100))
    else:
        print("אין נתונים זמינים להצגה.")

# --- טאב 3: גרף כמות לפי שנה (מצגת: Tab 3) ---
with out3:
    print("Vehicles count per Year:")
    if not df.empty and 'shnat_yitzur' in df.columns:
        plt.figure(figsize=(12, 6))
        year_counts = df['shnat_yitzur'].value_counts().sort_index()
        year_counts.plot(kind='bar', color='skyblue')
        plt.title('Number of cars produced per year')
        plt.xlabel('Year')
        plt.ylabel('Count')
        plt.show()
    else:
        print("אין נתונים זמינים לגרף או שהעמודה 'shnat_yitzur' אינה קיימת.")

# הצגת הממשק הסופי
display(tab)

מתחבר לענן ושולף נתונים...
שגיאה בעת משיכת נתונים מה-API: 404 Client Error: Not Found for url: https://data.gov.il/api/3/action/datastore_search?resource_id=053ad243-5e8b-4480-99c5-dc063856943d&limit=5000
שגיאה במשיכת נתונים מה-API, משתמשים בדאטה-סט לדוגמה.


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
